In [ ]:
import wandb
import numpy as np

api = wandb.Api()
ENTITY = "anon-entity"
PROJECT = "merlin-generation"

models = ['Qwen3-32B', 'medgemma-27b-it', 'Llama-3.3-70B-Instruct']
datasets = [f'cc_{s}' for s in ['abdominal_pain']] #'abdominal_pain', 'back_pain', 'chest_pain', 'cough', 'diarrhea', 'dyspnea', 'headache', 'vertigo'
metrics = ["v1_score", "v2_score", "v3_score", "v4_score"]

# Two storage containers
model_summary = {} # Model -> [4 Metrics x 4 Steps] (Averaged over 8 datasets)
raw_grid = {}      # Model -> Dataset -> [4 Metrics x 4 Steps] (Specific values)

for model_tag in models:
    # Intermediate storage to calculate means later
    # metric_name -> list of lists (8 datasets)
    model_accumulator = {m: [] for m in metrics}
    raw_grid[model_tag] = {}

    for ds_tag in datasets:
        filters = {"$and": [{"tags": model_tag}, {"tags": ds_tag}]}
        runs = list(api.runs(f"{ENTITY}/{PROJECT}", filters=filters))

        if not runs:
            raise ValueError(f"Missing run: {model_tag} + {ds_tag}")

        print(f'Found {len(runs)} for {model_tag} + {ds_tag}: {runs[0].name}')

        run = runs[0]
        history = run.history() # No keys filter, just like you found

        ds_results = []
        for m in metrics:
            series = history[m].dropna().values



            if len(series) < 4:
                raise ValueError(f"Run {run.name} ({ds_tag}) has only {len(series)} steps for {m}")

            # Align to 4 steps
            indices = np.linspace(0, len(series) - 1, 4).round().astype(int)
            aligned = series[indices].tolist()

            model_accumulator[m].append(aligned)
            ds_results.append(aligned)

        # Store the specific Dataset x Verifier matrix for this model
        raw_grid[model_tag][ds_tag.replace("cc_", "")] = ds_results

    # Calculate the Model x Verifier averages
    final_model_matrix = []
    for m in metrics:
        avg_v_score = np.array(model_accumulator[m]).mean(axis=0).tolist()
        final_model_matrix.append(avg_v_score)

    model_summary[model_tag] = final_model_matrix

print("✅ Data Aggregated!")

In [ ]:
import matplotlib.pyplot as plt
import os

from src.utils import init_notebook

plt.style.use('default')  # Standard Matplotlib light theme

save_path = "data/results/diagrams/generation_plots.pdf"
legend_position = "right"   # e.g. "upper right", "lower center", "center right", etc.
legend_fontsize = 8              # You can change this
figsize = (8, 3)                # For half-page / 2-column layout


# Create directory if it doesn't exist
os.makedirs(os.path.dirname(save_path), exist_ok=True)



In [ ]:
# Data
steps = [1, 2, 3, 4]
metrics = ["NormDot", "MRR", "MRR", "F1 Micro"]
titles = ["V1: Symptoms Extraction", "V2: Diagnoses Prediction", "V3: Lab-based Reranking", "V4: ICD Codes Prediction"]
data = model_summary

# Plot
fig, axes = plt.subplots(2, 2, figsize=figsize)
fig.subplots_adjust(hspace=0.3, wspace=0.2)

# Collect y-values for plots 2 and 3 to sync their scales
y_values_2_3 = []

for idx in [1, 2]:  # subplot 2 and 3
    for model in data:
        y_values_2_3.extend(data[model][idx])

y_min = min(y_values_2_3)
y_max = max(y_values_2_3)
# Add a small margin for visual comfort
margin = (y_max - y_min) * 0.1
y_limits = (y_min - margin, y_max + margin)


for idx, ax in enumerate(axes.flat):
    metric = metrics[idx]
    title = titles[idx]

    for model in data:
        ax.plot(steps, data[model][idx], marker='o', label=model)

    # Apply shared y-axis for subplots 2 and 3 specifically
    if idx in [1, 2]:
        ax.set_ylim(y_limits)
        # Use a Locator to ensure ticks don't crowd the squeezed space
        ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
    else:
        # This adds a 10% padding to the top and bottom automatically
        # for plots 1 and 4 based on THEIR specific data range.
        ax.margins(y=0.1)

    ax.set_title(title, fontsize=10)
    ax.set_ylabel(metric)
    ax.set_xticks(steps)
    ax.grid(True, linestyle='--', linewidth=0.5)

    if idx == 1:
        # Using bbox_to_anchor is safer for "squeezed" paper plots
        ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), fontsize=legend_fontsize)

# Optimization for paper space
plt.tight_layout(pad=1.0) # Explicit padding helps maintain your set limits

# Save as PDF
plt.tight_layout()
plt.savefig(save_path, format='pdf', bbox_inches="tight")
print(f"Saved plot to {save_path}")
plt.show()
